In [3]:
# -------- Config --------
DATADIR = "/work/gr-fe/bryan/data/TCGA"
COHORT = "TCGA-BRCA"
META_PICKLE = f"{DATADIR}/{COHORT}/02_processed/phenotype.processed.pkl"
OMICS = ["mRNA", "miRNA", "DNAm", "CNV", "RPPA"]  # datExpr_{omic}.csv for each
RAW_OMICS_DIR = f"{DATADIR}/{COHORT}/01_raw/"
OUT_DIR = f"{DATADIR}/{COHORT}/02_processed"
OVERWRITE = True  # overwrite existing output pickles

# -------- Imports --------
import os
import numpy as np
import pickle
from pathlib import Path
import pandas as pd

# -------- Helpers --------
def load_meta_ids(meta_path: str) -> pd.Series:
    """
    Load a pickle expected to contain either:
      - a pandas DataFrame with column 'ID', or
      - a dict with key 'ID' (array-like)
    Returns a clean string Series of IDs (duplicates removed, NaNs dropped).
    """
    # Try pandas first (faster for DataFrame pickles), fall back to pickle.load
    meta = None
    try:
        meta = pd.read_pickle(meta_path)
    except Exception:
        with open(meta_path, "rb") as f:
            meta = pickle.load(f)

    if isinstance(meta, pd.DataFrame) and "ID" in meta.columns:
        ids = meta["ID"]
    elif isinstance(meta, dict) and "ID" in meta:
        ids = pd.Series(meta["ID"], name="ID")
    else:
        raise ValueError("Meta pickle must be a DataFrame with 'ID' column or a dict with key 'ID'.")

    ids = ids.dropna().astype(str)
    ids = ids[~ids.duplicated()].reset_index(drop=True)
    return ids

def orient_to_ids(df: pd.DataFrame, id_set: set) -> pd.DataFrame | None:
    """
    Ensure sample IDs are in the index; if they're in columns, transpose.
    If neither index nor columns contain any meta IDs, return None.
    """
    # Normalize types to str for safe matching
    df.index = df.index.map(str)
    df.columns = df.columns.map(str)

    idx_hit = len(id_set.intersection(df.index))
    col_hit = len(id_set.intersection(df.columns))

    if idx_hit == 0 and col_hit > 0:
        df = df.T
        idx_hit = len(id_set.intersection(df.index))

    if idx_hit == 0:
        return None
    return df

# -------- Run --------
out_dir = Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

ids = load_meta_ids(META_PICKLE)
id_set = set(ids)

print(f"Loaded {len(ids)} unique meta IDs from: {META_PICKLE}")
print(f"Output dir: {out_dir}\n")

results = {}
for omic in OMICS:
    src = Path(RAW_OMICS_DIR) / f"datExpr_{omic}.csv"
    if not src.exists():
        print(f"[{omic}] SKIP - not found: {src}")
        continue

    try:
        df = pd.read_csv(src, index_col=0, dtype=str)
    except Exception as e:
        print(f"[{omic}] ERROR reading {src}: {e}")
        continue

    df_oriented = orient_to_ids(df, id_set)
    if df_oriented is None:
        print(f"[{omic}] SKIP - no overlap between meta IDs and {src.name} (rows or columns).")
        continue

    # Preserve meta ID order in the subset
    keep_ids = [i for i in ids if i in df_oriented.index]
    sub = df_oriented.loc[keep_ids].astype(np.float32) 

    out_path = out_dir / f"{omic}.pkl"
    if out_path.exists() and not OVERWRITE:
        print(f"[{omic}] Exists, not overwritten: {out_path} (shape={sub.shape})")
    else:
        with open(out_path, "wb") as f:
            pickle.dump({"expr": sub}, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"[{omic}] Saved {sub.shape} to {out_path}")

    results[omic] = sub

# Optional: quick peek at one modality
for omic in OMICS:
    if omic in results:
        display(results[omic].head())
        break


Loaded 1096 unique meta IDs from: /work/gr-fe/bryan/data/TCGA/TCGA-BRCA/02_processed/phenotype.processed.pkl
Output dir: /work/gr-fe/bryan/data/TCGA/TCGA-BRCA/02_processed

[mRNA] Saved (1081, 60660) to /work/gr-fe/bryan/data/TCGA/TCGA-BRCA/02_processed/mRNA.pkl
[miRNA] Saved (1063, 1881) to /work/gr-fe/bryan/data/TCGA/TCGA-BRCA/02_processed/miRNA.pkl
[DNAm] Saved (778, 200000) to /work/gr-fe/bryan/data/TCGA/TCGA-BRCA/02_processed/DNAm.pkl
[CNV] Saved (1051, 60265) to /work/gr-fe/bryan/data/TCGA/TCGA-BRCA/02_processed/CNV.pkl
[RPPA] Saved (867, 464) to /work/gr-fe/bryan/data/TCGA/TCGA-BRCA/02_processed/RPPA.pkl


,ENSG00000000003.15,ENSG00000000005.6,ENSG00000000419.13,ENSG00000000457.14,ENSG00000000460.17,ENSG00000000938.13,ENSG00000000971.16,ENSG00000001036.14,ENSG00000001084.13,ENSG00000001167.14,...,ENSG00000288661.1,ENSG00000288662.1,ENSG00000288663.1,ENSG00000288665.1,ENSG00000288667.1,ENSG00000288669.1,ENSG00000288670.1,ENSG00000288671.1,ENSG00000288674.1,ENSG00000288675.1
TCGA-E2-A1IU,850.0,5.0,1680.0,1559.0,402.0,334.0,1271.0,1713.0,3670.0,2376.0,...,0.0,0.0,12.0,0.0,0.0,0.0,231.0,0.0,3.0,6.0
TCGA-A1-A0SB,4148.0,604.0,1194.0,996.0,337.0,111.0,9044.0,788.0,1417.0,2324.0,...,0.0,0.0,24.0,0.0,0.0,0.0,218.0,0.0,11.0,26.0
TCGA-A2-A04W,2718.0,1.0,1914.0,677.0,291.0,224.0,1053.0,2223.0,703.0,1271.0,...,0.0,0.0,9.0,0.0,0.0,0.0,121.0,0.0,1.0,34.0
TCGA-AN-A0AM,2501.0,1.0,7444.0,2268.0,1177.0,404.0,3334.0,29363.0,2136.0,1815.0,...,0.0,1.0,23.0,0.0,1.0,0.0,432.0,0.0,11.0,21.0
TCGA-LL-A440,2842.0,113.0,1617.0,1358.0,408.0,5354.0,5237.0,2628.0,2424.0,1980.0,...,0.0,0.0,69.0,0.0,0.0,0.0,332.0,0.0,28.0,46.0
